In [1]:
import os
os.environ["PYTHONNOUSERSITE"] = "1"

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
from adaptive_snapkv.monkeypatch.monkeypatch import replace_llama_adaptive

replace_llama_adaptive()

model_id = "meta-llama/Meta-Llama-3-8B"
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)

# AdaKV settings (same as before)
config.window_size = 32
config.base_capacity = 512
config.kernel_size = 7
config.pooling = "maxpool"
config.floor_alpha = 0.5
config.pyram_mode = False
config.pyram_beta = 20
# Missing attributes required by AdaKV
config.skip = False
config.normalize = False
config.gqa_support = True
config.gqa_func = "mean"   # 'max' or 'mean'
config.full_kv_scoring = False


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=quantization_config,
    attn_implementation="flash_attention_2",   # official AdaKV uses flash_attn
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

prompt = "Explain the theory of general relativity in simple terms."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# No need to pass device_map – the 4-bit model will load on GPU 0 automatically

/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/c/Users/User/Desktop/work/UOS/Updating/LiteratureAI_v3/runs/AdaKV-entropy/adaptive_snapkv/monkeypatch/monkeypatch.py:30: UserWarning: Transformers version 4.45.0 might not be compatible with SnapKV. SnapKV is tested with Transformers version ['4.37'].
  warnings.warn(f"Transformers version {transformers_version} might not be compatible with SnapKV. SnapKV is tested with Transformers version {version_list}.")
`low_cpu_mem_usage` was None, now set to True since model is quantized.
Loading checkpoint shards: 100%|██████████| 4/4 [02:07<00:00, 31.95s/it]


In [2]:
import time
import torch

def evaluate_generation(model, tokenizer, prompt, max_new_tokens=100):
    input = tokenizer(prompt, truncation=False, return_tensors="pt").to("cuda")
    context_length = input.input_ids.shape[-1]

    torch.cuda.synchronize()
    t0 = time.time()
    output = model.generate(
        **input,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        eos_token_id=[tokenizer.eos_token_id],
    )[0]
    torch.cuda.synchronize()
    t = time.time() - t0

    generated = tokenizer.decode(output[context_length:], skip_special_tokens=True)
    tokens_per_sec = max_new_tokens / t
    return generated, t, tokens_per_sec


In [3]:
import json, time, torch, gc
from datasets import load_dataset
from tqdm import tqdm

def evaluate_longbench(model, tokenizer, dataset_name="qasper", 
                       max_length=4096, max_new_tokens=128, out_path=None):
    """Run one LongBench dataset on the already-loaded model."""
    dataset2prompt = json.load(open("experiments/LongBench/config/dataset2prompt.json"))
    dataset2maxlen = json.load(open("experiments/LongBench/config/dataset2maxlen.json"))
    
    data = load_dataset("THUDM/LongBench", dataset_name, split="test")
    prompt_format = dataset2prompt[dataset_name]
    max_gen = dataset2maxlen[dataset_name]
    
    preds = []
    for json_obj in tqdm(data):
        prompt = prompt_format.format(**json_obj)
        tokenized = tokenizer(prompt, truncation=False, return_tensors="pt").input_ids[0]
        
        if len(tokenized) > max_length:
            half = max_length // 2
            prompt = (tokenizer.decode(tokenized[:half], skip_special_tokens=True) + 
                      tokenizer.decode(tokenized[-half:], skip_special_tokens=True))
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        ctx_len = inputs.input_ids.shape[-1]
        
        with torch.no_grad():
            output = model.generate(
                **inputs, max_new_tokens=max_gen,
                do_sample=False, temperature=1.0,
                eos_token_id=[tokenizer.eos_token_id],
            )[0]
        
        pred = tokenizer.decode(output[ctx_len:], skip_special_tokens=True)
        preds.append({"pred": pred, "answers": json_obj["answers"]})
        
        gc.collect()
        torch.cuda.empty_cache()
    
    if out_path:
        with open(out_path, "w") as f:
            for p in preds:
                json.dump(p, f, ensure_ascii=False)
                f.write("\n")
    
    return preds

In [4]:
import re, string, json, os, numpy as np
from collections import Counter
from rouge import Rouge

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        return "".join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    common = Counter(prediction) & Counter(ground_truth)
    num_same = sum(common.values())
    if num_same == 0: return 0
    precision = 1.0 * num_same / len(prediction)
    recall = 1.0 * num_same / len(ground_truth)
    return (2 * precision * recall) / (precision + recall)

def qa_f1_score(prediction, ground_truth, **kwargs):
    return f1_score(normalize_answer(prediction).split(), normalize_answer(ground_truth).split())

def rouge_score(prediction, ground_truth, **kwargs):
    try:
        return Rouge().get_scores([prediction], [ground_truth], avg=True)["rouge-l"]["f"]
    except:
        return 0.0

dataset2metric = {
    "narrativeqa": qa_f1_score, "qasper": qa_f1_score,
    "multifieldqa_en": qa_f1_score, "hotpotqa": qa_f1_score,
    "2wikimqa": qa_f1_score, "musique": qa_f1_score,
    "gov_report": rouge_score, "qmsum": rouge_score,
    "multi_news": rouge_score, "samsum": rouge_score,
}

def score_dataset(jsonl_path):
    with open(jsonl_path) as f:
        data = [json.loads(line) for line in f]
    dataset = os.path.splitext(os.path.basename(jsonl_path))[0]
    metric = dataset2metric.get(dataset, qa_f1_score)
    scores = []
    for item in data:
        best = 0
        for gt in item["answers"]:
            best = max(best, metric(item["pred"], gt))
        scores.append(best)
    return dataset, round(100 * np.mean(scores), 2)



In [5]:
import json, time, torch, gc, os
from datasets import load_dataset
from tqdm import tqdm

dataset2prompt = json.load(open("experiments/LongBench/config/dataset2prompt.json"))
dataset2maxlen = json.load(open("experiments/LongBench/config/dataset2maxlen.json"))

datasets = ["narrativeqa", "qasper", "hotpotqa", "gov_report", "multi_news", "samsum"]
max_length = 4096

def get_preds(model, tokenizer, dataset_name, out_path):
    data = load_dataset("THUDM/LongBench", dataset_name, split="test")
    prompt_format = dataset2prompt[dataset_name]
    max_gen = dataset2maxlen[dataset_name]

    preds, times = [], []
    for json_obj in tqdm(data, desc=dataset_name):
        prompt = prompt_format.format(**json_obj)
        tokenized = tokenizer(prompt, truncation=False, return_tensors="pt").input_ids[0]
        if len(tokenized) > max_length:
            half = max_length // 2
            prompt = (tokenizer.decode(tokenized[:half], skip_special_tokens=True) +
                      tokenizer.decode(tokenized[-half:], skip_special_tokens=True))

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        ctx_len = inputs.input_ids.shape[-1]

        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            output = model.generate(
                **inputs, max_new_tokens=max_gen,
                do_sample=False, temperature=1.0,
                eos_token_id=[tokenizer.eos_token_id],
            )[0]
        torch.cuda.synchronize()
        dt = time.time() - t0
        times.append(dt)

        pred = tokenizer.decode(output[ctx_len:], skip_special_tokens=True)
        preds.append({"pred": pred, "answers": json_obj["answers"],
                      "all_classes": json_obj["all_classes"], "length": json_obj["length"]})

        gc.collect()
        torch.cuda.empty_cache()

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        for p in preds:
            json.dump(p, f, ensure_ascii=False)
            f.write("\n")

    return np.mean(times)

In [6]:
from tqdm import tqdm

def evaluate_longbench_batched(model, tokenizer, dataset_name, max_length=4096,
                                max_new_tokens=128, batch_size=4, out_path=None):
    dataset2prompt = json.load(open("experiments/LongBench/config/dataset2prompt.json"))
    dataset2maxlen = json.load(open("experiments/LongBench/config/dataset2maxlen.json"))

    data_all = [d for d in load_dataset("THUDM/LongBench", dataset_name, split="test")]
    prompt_format = dataset2prompt[dataset_name]
    max_gen = dataset2maxlen[dataset_name]

    preds = []
    for i in tqdm(range(0, len(data_all), batch_size), desc=dataset_name):
        batch = data_all[i:i+batch_size]
        prompts = []
        for json_obj in batch:
            prompt = prompt_format.format(**json_obj)
            tokenized = tokenizer(prompt, truncation=False, return_tensors="pt").input_ids[0]
            if len(tokenized) > max_length:
                half = max_length // 2
                prompt = (tokenizer.decode(tokenized[:half], skip_special_tokens=True) +
                          tokenizer.decode(tokenized[-half:], skip_special_tokens=True))
            prompts.append(prompt)

        inputs = tokenizer(prompts, padding=True, return_tensors="pt").to("cuda")
        ctx_lens = inputs.attention_mask.sum(dim=-1)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_gen,
                do_sample=False, temperature=1.0,
                eos_token_id=[tokenizer.eos_token_id],
            )

        for j, output in enumerate(outputs):
            pred = tokenizer.decode(output[ctx_lens[j]:], skip_special_tokens=True)
            preds.append({"pred": pred, "answers": batch[j]["answers"],
                          "all_classes": batch[j]["all_classes"], "length": batch[j]["length"]})

        gc.collect()
        torch.cuda.empty_cache()

    if out_path:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        with open(out_path, "w") as f:
            for p in preds:
                json.dump(p, f, ensure_ascii=False)
                f.write("\n")
    return preds



def get_preds_batched(model, tokenizer, dataset_name, out_path, batch_size=4):
    data_all = [d for d in load_dataset("THUDM/LongBench", dataset_name, split="test")]
    prompt_format = dataset2prompt[dataset_name]
    data_all.sort(key=lambda x: len(prompt_format.format(**x)))
    max_gen = dataset2maxlen[dataset_name]

    preds, times = [], []
    for i in tqdm(range(0, len(data_all), batch_size), desc=dataset_name):
        batch = data_all[i:i+batch_size]
        prompts = []
        for json_obj in batch:
            prompt = prompt_format.format(**json_obj)
            tokenized = tokenizer(prompt, truncation=False, return_tensors="pt").input_ids[0]
            if len(tokenized) > max_length:
                half = max_length // 2
                prompt = (tokenizer.decode(tokenized[:half], skip_special_tokens=True) +
                          tokenizer.decode(tokenized[-half:], skip_special_tokens=True))
            prompts.append(prompt)

        inputs = tokenizer(prompts, padding=True, return_tensors="pt").to("cuda")
        ctx_lens = inputs.attention_mask.sum(dim=-1)

        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_gen,
                do_sample=False, temperature=1.0,
                eos_token_id=[tokenizer.eos_token_id],
            )
        torch.cuda.synchronize()
        dt = time.time() - t0
        times.append(dt)

        for j, output in enumerate(outputs):
            pred = tokenizer.decode(output[ctx_lens[j]:], skip_special_tokens=True)
            preds.append({"pred": pred, "answers": batch[j]["answers"],
                          "all_classes": batch[j]["all_classes"],
                          "length": batch[j]["length"]})

        gc.collect()
        torch.cuda.empty_cache()

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        for p in preds:
            json.dump(p, f, ensure_ascii=False)
            f.write("\n")

    return np.mean(times)

In [ ]:

# Vanilla Llama
import transformers.models.llama.modeling_llama as llama_models
_vanilla_model_forward = llama_models.LlamaModel.forward
_vanilla_attn_forward = llama_models.LlamaFlashAttention2.forward

import importlib
import types
import transformers.models.llama.modeling_llama as llama_models


# Reload to restore original class methods
llama_models = importlib.reload(llama_models)

# Swap model to vanilla
model.model.forward = types.MethodType(
    llama_models.LlamaModel.forward, model.model
)

# Swap ALL attention layer forwards
for layer in model.model.layers:
    attn = layer.self_attn
    # Restore vanilla class method
    attn.forward = types.MethodType(
        llama_models.LlamaFlashAttention2.forward, attn
    )


out_vanilla = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(out_vanilla[0]))

base_gen_text, base_gen_time, base_tok_s = evaluate_generation(model, tokenizer, prompt)
print(f"Generated ({base_tok_s:.1f} tok/s):\n{base_gen_text}")

import gc
gc.collect()
torch.cuda.empty_cache()

base_pred = evaluate_longbench_batched(model, tokenizer, dataset_name="qasper", max_length=4096, max_new_tokens=128, out_path="qasper_preds.jsonl", batch_size=3)

import gc
gc.collect()
torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


<|begin_of_text|>Explain the theory of general relativity in simple terms. I don't understand it. What is a black hole? What is the event horizon? Why is it a black hole? Why do we say that the event horizon is the point of no return? Why is time warped? What is the difference between
Generated (17.0 tok/s):
 What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications of this theory? What are the implications


/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
qasper:   0%|          | 0/67 [00:00<?, ?it/s]/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being 

KeyboardInterrupt: 

In [9]:
import types
import torch
from transformers.cache_utils import Cache as HFCache
import adaptive_snapkv.monkeypatch.adaptive_llama_hijack as alh
import transformers.models.llama.modeling_llama as llama_models

import gc
gc.collect()
torch.cuda.empty_cache()

import adaptive_snapkv.monkeypatch.adaptive_llama_hijack as alh
model.model.config.base_capacity = 512

model.model.forward = types.MethodType(
    alh.adaptive_LlamaModel_forward, model.model
)
for layer in model.model.layers:
    attn = layer.self_attn
    attn.forward = types.MethodType(
        alh.adaptive_llama_flash_attn2_forward, attn
    )

model.model.config.base_capacity = 512

print("✓ Cache init fix applied")


out_ada = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(out_ada[0]))

# ada_gen_text, ada_gen_time, ada_tok_s = evaluate_generation(model, tokenizer, prompt)
# print(f"Generated ({ada_tok_s:.1f} tok/s):\n{ada_gen_text}")

ada_pred = evaluate_longbench(model, tokenizer, dataset_name="qasper", max_length=4096, max_new_tokens=128, out_path="pred/adakv_run/qasper_preds.jsonl")

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


✓ Cache init fix applied
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True


/mnt/c/Users/User/Desktop/work/UOS/Updating/LiteratureAI_v3/runs/AdaKV-entropy/adaptive_snapkv/monkeypatch/snapkv_utils.py:787: UserWarning: GQA currently supports only for mistral-7B-v0.2 model
  warnings.warn("GQA currently supports only for mistral-7B-v0.2 model")


Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, 

/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
100%|██████████| 200/200 [21:21<00:00,  6.41s/it]


In [8]:
from adaptive_snapkv.monkeypatch.monkeypatch import config_compress

head_entropy_dir = "pred/head_entropy_run/"

model.model.config.use_entropy = True
model.model.config.entropy_alpha = 2.0


# head_entropy = model.generate(**inputs, max_new_tokens=50)
# print(tokenizer.decode(head_entropy[0]))

import gc 
gc.collect()
torch.cuda.empty_cache()


#base_pred = evaluate_longbench(model, tokenizer, dataset_name="qasper", max_length=4096, max_new_tokens=128, out_path="qasper_preds.jsonl")
#head_entropy_pred = evaluate_longbench(model, tokenizer, dataset_name="qasper", max_length=4096, max_new_tokens=128, out_path=head_entropy_dir + "qasper_preds.jsonl")

head_entropy_pred = evaluate_longbench_batched(model, tokenizer, dataset_name="qasper", max_length=4096, max_new_tokens=128, batch_size=3, out_path=head_entropy_dir + "qasper_preds.jsonl")

/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
qasper:   0%|          | 0/67 [00:00<?, ?it/s]/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being 

In [7]:
print("=== Vanilla ===")
# swap to vanilla (reload + re-patch)
import importlib, types
import transformers.models.llama.modeling_llama as llama_models

dataset2prompt = json.load(open("experiments/LongBench/config/dataset2prompt.json"))
dataset2maxlen = json.load(open("experiments/LongBench/config/dataset2maxlen.json"))
datasets = ["narrativeqa"#, "qasper", "hotpotqa", "gov_report", "multi_news", "samsum"
            ]
max_length = 4096

for budget in [512, 256, 128]:

    model.model.config.base_capacity = budget
    model.model.config.use_entropy = False
    llama_models = importlib.reload(llama_models)
    model.model.forward = types.MethodType(llama_models.LlamaModel.forward, model.model)
    for layer in model.model.layers:
        c = layer.self_attn.__class__
        vanilla_fwd = getattr(llama_models, c.__name__).forward
        layer.self_attn.forward = types.MethodType(vanilla_fwd, layer.self_attn)

    for ds in datasets:
        base_dir = f"pred/sweep/b{budget}/vanilla_run/{ds}.jsonl"
        get_preds(model, tokenizer, ds, base_dir)

    import gc
    gc.collect()
    torch.cuda.empty_cache()

    
    # # ── AdaKV run ──
    print("\n=== AdaKV ===")
    import adaptive_snapkv.monkeypatch.adaptive_llama_hijack as alh
    alh = importlib.reload(alh)
    llama_models.LlamaModel.forward = alh.adaptive_LlamaModel_forward
    model.model.forward = types.MethodType(alh.adaptive_LlamaModel_forward, model.model)
    for layer in model.model.layers:
        c = layer.self_attn.__class__
        ada_fwd = getattr(alh, f"adaptive_{c.__name__.lower()}_forward",
                        alh.adaptive_llama_flash_attn2_forward)
        layer.self_attn.forward = types.MethodType(ada_fwd, layer.self_attn)



    model.model.config.use_entropy = False
    for ds in datasets:
        ada_kv_dir = f"pred/sweep/b{budget}/adakv_run/{ds}.jsonl"
        get_preds(model, tokenizer, ds, ada_kv_dir)

    import gc
    gc.collect()
    torch.cuda.empty_cache()

    # -- Entropy run (same as AdaKV but with skip=True to disable updates) --
    print("\n=== Entropy (AdaKV with skip=True) ===")
    model.model.config.use_entropy = True
    model.model.config.skip = False
    model.model.config.full_kv_scoring = False
    for layer in model.model.layers:
        if hasattr(layer.self_attn, 'kv_cluster'):
            del layer.self_attn.kv_cluster
    for ds in datasets:
        entropy_dir = f"pred/sweep/b{budget}/entropy_run/{ds}.jsonl"
        get_preds(model, tokenizer, ds, entropy_dir)
    
    

print("\nDone! Results in pred/vanilla_run/ and pred/adakv_run/")

=== Vanilla ===


/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
narrativeqa:   0%|          | 0/200 [00:00<?, ?it/s]/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Starting from v4.46, the `logit

KeyboardInterrupt: 

In [ ]:



datasets = [ #"narrativeqa", 
             "qasper", "hotpotqa", "gov_report", "multi_news", "samsum"
            ]
for BUDGET in [256, 128, 64]:

    model.model.config.base_capacity = BUDGET
    model.model.config.full_kv_scoring = False

    # ── AdaKV ──
    model.model.config.use_entropy = False
    for layer in model.model.layers:
        if hasattr(layer.self_attn, 'kv_cluster'):
            del layer.self_attn.kv_cluster

    for ds in datasets:
        path = f"pred/sweep/b{BUDGET}/adakv/{ds}.jsonl"
        if os.path.exists(path):
            print(f"  {ds}: exists, skip")
            continue
        get_preds(model, tokenizer, ds, path)
    

    gc.collect()
    torch.cuda.empty_cache()

    # ── Entropy ──
    model.model.config.use_entropy = True
    for layer in model.model.layers:
        if hasattr(layer.self_attn, 'kv_cluster'):
            del layer.self_attn.kv_cluster

    for ds in datasets:
      path = f"pred/sweep/b{BUDGET}/entropy/{ds}.jsonl"
      if os.path.exists(path):
            print(f"  {ds}: exists, skip")
            continue
      get_preds(model, tokenizer, ds, path)

    print(f"✅ Budget {BUDGET} done")

  qasper: exists, skip
  hotpotqa: exists, skip
  gov_report: exists, skip
  multi_news: exists, skip
  samsum: exists, skip
  qasper: exists, skip
  hotpotqa: exists, skip
  gov_report: exists, skip
  multi_news: exists, skip
  samsum: exists, skip
✅ Budget 256 done
  qasper: exists, skip
  hotpotqa: exists, skip


/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
gov_report:   0%|          | 0/200 [00:00<?, ?it/s]/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
We detected that you are passing

Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=False, entropy_computation=window


/mnt/c/Users/User/Desktop/work/UOS/Updating/LiteratureAI_v3/runs/AdaKV-entropy/adaptive_snapkv/monkeypatch/snapkv_utils.py:769: UserWarning: GQA currently supports only for mistral-7B-v0.2 model
  warnings.warn("GQA currently supports only for mistral-7B-v0.2 model")


Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=False, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=False, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=False, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=False, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=False, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
gov_report: 100%|██████████| 200/200 [1:08:06<00:00, 20.43s/it]
/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
qasper:   0%|          | 0/200 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=True, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=True, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=True, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=True, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, use_entropy=True, entropy_computation=window
Compress config(Ada): window_size=32, base_capacity=96, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode

gov_report:  36%|███▋      | 73/200 [29:36<1:01:21, 28.99s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [10]:
import gc
gc.collect()
import torch
torch.cuda.empty_cache()



In [7]:
import json, time, torch, gc, os, numpy as np
from datasets import load_dataset
from tqdm import tqdm

dataset2prompt = json.load(open("experiments/LongBench/config/dataset2prompt.json"))
dataset2maxlen = json.load(open("experiments/LongBench/config/dataset2maxlen.json"))
datasets = ["narrativeqa"#, "qasper", "hotpotqa", "gov_report", "multi_news", "samsum"
            ]
max_length = 4096

# def get_preds(model, tokenizer, dataset_name, out_path):
#     data_all = [d for d in load_dataset("THUDM/LongBench", dataset_name, split="test")]
#     prompt_format = dataset2prompt[dataset_name]
#     max_gen = dataset2maxlen[dataset_name]

#     preds = []
#     for json_obj in tqdm(data_all, desc=dataset_name):
        
#         prompt = prompt_format.format(**json_obj)
#         tokenized = tokenizer(prompt, truncation=False, return_tensors="pt").input_ids[0]
#         if len(tokenized) > max_length:
#             half = max_length // 2
#             prompt = (tokenizer.decode(tokenized[:half], skip_special_tokens=True) +
#                       tokenizer.decode(tokenized[-half:], skip_special_tokens=True))

#         inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
#         ctx_len = inputs.input_ids.shape[-1]

#         with torch.no_grad():
#             output = model.generate(
#                 **inputs, max_new_tokens=max_gen,
#                 do_sample=False, temperature=1.0,
#                 eos_token_id=[tokenizer.eos_token_id],
#             )[0]

#         pred = tokenizer.decode(output[ctx_len:], skip_special_tokens=True)
#         preds.append({"pred": pred, "answers": json_obj["answers"],
#                       "all_classes": json_obj["all_classes"], "length": json_obj["length"]})

#         gc.collect()
#         torch.cuda.empty_cache()

#     os.makedirs(os.path.dirname(out_path), exist_ok=True)
#     with open(out_path, "w") as f:
#         for p in preds:
#             json.dump(p, f, ensure_ascii=False)
#             f.write("\n")

# ─────────────────────────────────────
# Sweep: budgets [512, 256, 128], modes [vanilla, adakv, entropy]
# ─────────────────────────────────────
import importlib, types
import transformers.models.llama.modeling_llama as llama_models
import adaptive_snapkv.monkeypatch.adaptive_llama_hijack as alh

for budget in [512, 256, 128]:
    model.model.config.base_capacity = budget
    model.model.config.full_kv_scoring = False  # ← VRAM safe
    
    for mode_name, mode_cfg in [
        ("vanilla", {"use_entropy": False, "forward_fn": None}),  # None = vanilla
        ("adakv",   {"use_entropy": False}),
        ("entropy", {"use_entropy": True}),
    ]:
        print(f"\n{'='*50}\n{budget=} {mode_name=}\n{'='*50}")
        for layer in model.model.layers:
            if hasattr(layer.self_attn, 'kv_cluster'):
                del layer.self_attn.kv_cluster
        gc.collect()
        torch.cuda.synchronize()
        # Swap forward mode
        if mode_name == "vanilla":
            llama_models = importlib.reload(llama_models)
            model.model.forward = types.MethodType(llama_models.LlamaModel.forward, model.model)
            for layer in model.model.layers:
                attn = layer.self_attn
                attn.forward = types.MethodType(llama_models.LlamaFlashAttention2.forward, attn)
        else:
            alh = importlib.reload(alh)
            model.model.config.use_entropy = mode_cfg["use_entropy"]
            llama_models.LlamaModel.forward = alh.adaptive_LlamaModel_forward
            model.model.forward = types.MethodType(alh.adaptive_LlamaModel_forward, model.model)
            for layer in model.model.layers:
                c = layer.self_attn.__class__
                ada_fwd = getattr(alh, f"adaptive_{c.__name__.lower()}_forward",
                                 alh.adaptive_llama_flash_attn2_forward)
                layer.self_attn.forward = types.MethodType(ada_fwd, layer.self_attn)
            # Fresh clusters
            for layer in model.model.layers:
                if hasattr(layer.self_attn, 'kv_cluster'):
                    del layer.self_attn.kv_cluster
            gc.collect()
            torch.cuda.synchronize()

        torch.cuda.empty_cache()
        
        for ds in datasets:
            out_path = f"pred/sweep/b{budget}/{mode_name}/{ds}.jsonl"
            if os.path.exists(out_path):
                print(f"  {ds}: exists, skip")
                continue
            try:
                get_preds(model, tokenizer, ds, out_path)
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"  {ds}: OOM, skip")
                    torch.cuda.empty_cache()
                else:
                    raise

print("\n✅ Sweep complete!")


budget=512 mode_name='vanilla'


/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/datasets/load.py:1491: FutureWarning: The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
narrativeqa:   0%|          | 0/200 [00:00<?, ?it/s]/home/kashora/miniconda3/envs/adakv_final/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Starting from v4.46, the `logit

KeyboardInterrupt: 

In [8]:

import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import adaptive_snapkv.monkeypatch.adaptive_llama_hijack as alh
alh = importlib.reload(alh)
llama_models.LlamaModel.forward = alh.adaptive_LlamaModel_forward
model.model.forward = types.MethodType(alh.adaptive_LlamaModel_forward, model.model)
for layer in model.model.layers:
    c = layer.self_attn.__class__
    ada_fwd = getattr(alh, f"adaptive_{c.__name__.lower()}_forward",
                     alh.adaptive_llama_flash_attn2_forward)
    layer.self_attn.forward = types.MethodType(ada_fwd, layer.self_attn)


model.model.config.base_capacity = 512

for ds in datasets:
    get_preds(model, tokenizer, ds, f"pred/vanilla_run/{ds}.jsonl")

narrativeqa:   0%|          | 0/200 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pool

narrativeqa:   0%|          | 0/200 [00:41<?, ?it/s]


KeyboardInterrupt: 

In [12]:
print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"Free:      {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved())/1e9:.2f} GB")

Allocated: 6.20 GB
Reserved:  13.75 GB
Free:      -0.87 GB


In [ ]:

import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import os

vanilla_dir = "pred/vanilla_run/"
adakv_dir = "pred/adakv_run/"
head_entropy_dir = "pred/head_entropy_run/"

results = []
for fname in os.listdir(vanilla_dir):
    if not fname.endswith(".jsonl"):
        continue
    
    ds = fname.replace(".jsonl", "")
    van_path = os.path.join(vanilla_dir, fname)
    ada_path = os.path.join(adakv_dir, fname)
    
    _, van_score = score_dataset(van_path)
    
    if os.path.exists(ada_path):
        _, ada_score = score_dataset(ada_path)
        delta = ada_score - van_score
    else:
        ada_score = None
        delta = None
    
    results.append((ds, van_score, ada_score, delta))

# Print comparison table
print(f"{'Dataset':<20} {'Vanilla':>8} {'AdaKV':>8} {'Δ':>8}")
print("-" * 48)
van_scores = []
for ds, v, a, d in sorted(results, key=lambda x: x[3] if x[3] is not None else 0):
    van_scores.append(v)
    a_str = f"{a:>8.2f}" if a is not None else "   N/A  "
    d_str = f"{d:>+8.2f}" if d is not None else "   N/A  "
    print(f"{ds:<20} {v:>8.2f} {a_str} {d_str}")

print("-" * 48)
avg_v = np.mean(van_scores)
completed = [r for r in results if r[2] is not None]
avg_a = np.mean([r[2] for r in completed]) if completed else 0
print(f"{'AVERAGE':<20} {avg_v:>8.2f} {'N/A':>8}" if not completed else
      f"{'AVERAGE':<20} {avg_v:>8.2f} {avg_a:>8.2f} {avg_a-avg_v:>+8.2f}")

In [ ]:
import torch
import torch.nn.functional as F
import adaptive_snapkv.monkeypatch.snapkv_utils as su

_original_calcul_attn_sore = su.AdaptiveSnapKVCluster.calcul_attn_sore

def patched_calcul_attn_sore(self, key_states, query_states):
    bsz, num_heads, q_len, head_dim = query_states.shape
    
    if self.full_kv_scoring:
        q_slice = query_states
    else:
        q_slice = query_states[..., -self.window_size:, :]
    
    attn_weights = torch.matmul(q_slice, key_states.transpose(2, 3)) / math.sqrt(head_dim)
    
    n_q = attn_weights.shape[-2]
    n_k = min(attn_weights.shape[-1], self.window_size)
    
    # Fixed causal mask — broadcasts to [n_q, n_k]
    mask = torch.full((n_q, n_k), torch.finfo(attn_weights.dtype).min,
                      device=attn_weights.device)
    row_idx = torch.arange(n_q, device=attn_weights.device).unsqueeze(1)
    col_idx = torch.arange(n_k, device=attn_weights.device).unsqueeze(0)
    mask.masked_fill_(col_idx <= (row_idx + n_k - n_q), 0)
    
    attention_mask = mask[None, None, :, :]
    attn_weights[:, :, -n_q:, -n_k:] += attention_mask
    
    attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
    
    if self.use_entropy:
        p_alpha = attn_weights ** self.entropy_alpha
        sum_p_alpha = p_alpha.sum(dim=-1)
        head_entropy = torch.log(sum_p_alpha + 1e-10) / (1 - self.entropy_alpha)
        head_entropy = head_entropy.mean(dim=-1)
    
    attn_weights_mean = attn_weights[:, :, -self.window_size:, : -self.window_size]
    if attn_weights_mean.shape[-1] == 0:
        empty = torch.zeros(bsz, num_heads, 0, device=attn_weights.device, dtype=attn_weights.dtype)
        return (empty, head_entropy) if self.use_entropy else empty
    attn_weights_mean = attn_weights_mean.mean(dim=-2)
    
    if self.gqa_support:
        attn_weights_mean = attn_weights_mean.view(bsz, num_heads // self.num_key_value_groups, self.num_key_value_groups, -1)
        if self.gqa_func == 'max':
            attn_weights_mean = attn_weights_mean.max(dim=-2).values
        elif self.gqa_func == 'mean':
            attn_weights_mean = attn_weights_mean.mean(dim=-2)
        if self.use_entropy:
            he = head_entropy.view(bsz, -1, self.num_key_value_groups)
            head_entropy = he.max(dim=-1).values if self.gqa_func == 'max' else he.mean(dim=-1)
    
    if attn_weights_mean.shape[-1] >= self.kernel_size:
        if self.pooling == 'avgpool':
            attn_weights_mean = F.avg_pool1d(attn_weights_mean, kernel_size=self.kernel_size,
                                              padding=self.kernel_size // 2, stride=1)
        elif self.pooling == 'maxpool':
            attn_weights_mean = F.max_pool1d(attn_weights_mean, kernel_size=self.kernel_size,
                                              padding=self.kernel_size // 2, stride=1)
    
    if self.use_entropy:
        return attn_weights_mean, head_entropy
    return attn_weights_mean

# Apply hot-patch at the class level (affects all future forward calls)
su.AdaptiveSnapKVCluster.calcul_attn_sore = patched_calcul_attn_sore

# Also patch existing cluster instances
for layer in model.model.layers:
    if hasattr(layer.self_attn, 'kv_cluster'):
        layer.self_attn.kv_cluster.calcul_attn_sore = patched_calcul_attn_sore.__get__(
            layer.self_attn.kv_cluster, su.AdaptiveSnapKVCluster
        )



print("✓ calcul_attn_sore hot-patched with fixed causal mask")

✓ calcul_attn_sore hot-patched with fixed causal mask


In [13]:
import torch
import adaptive_snapkv.monkeypatch.snapkv_utils as su
import math 
# Enable entropy mode
model.model.config.use_entropy = True
model.model.config.entropy_alpha = 2.0
model.model.config.full_kv_scoring = True

# Need to reinitialize clusters (force new kv_cluster instances)
# Easiest: delete existing clusters so init_adaptive_snapkv re-creates them
for layer in model.model.layers:
    if hasattr(layer.self_attn, 'kv_cluster'):
        del layer.self_attn.kv_cluster

# Run one forward with a long-ish prompt
prompt = "The theory of relativity " * 40
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model(**inputs, use_cache=True)

# Check layer 0
attn0 = model.model.layers[0].self_attn
print(f"Layer 0 kv_cluster exists: {hasattr(attn0, 'kv_cluster')}")
if hasattr(attn0, 'kv_cluster'):
    print(f"use_entropy: {attn0.kv_cluster.use_entropy}")
    # We can't directly access entropy (it was local to calcul_attn_sore)
    # But we CAN see the resulting head budgets
    print(f"Head lengths: {attn0.kv_cluster.head_lens}")
    print(f"Total budget: {attn0.kv_cluster.klen_sum}")

Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=

In [14]:
def capture_head_lens(model, use_entropy):
    model.model.config.use_entropy = use_entropy
    for layer in model.model.layers:
        if hasattr(layer.self_attn, 'kv_cluster'):
            del layer.self_attn.kv_cluster
    
    with torch.no_grad():
        model(**inputs, use_cache=True)
    
    return {i: layer.self_attn.kv_cluster.head_lens.tolist() 
            for i, layer in enumerate(model.model.layers)
            if hasattr(layer.self_attn, 'kv_cluster')}

heads_ada = capture_head_lens(model, use_entropy=False)
# Clear and re-run
heads_entropy = capture_head_lens(model, use_entropy=True)

# Compare layer 0
print(f"AdaKV budgets layer 0:   {heads_ada[0]}")
print(f"Entropy budgets layer 0: {heads_entropy[0]}")


Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=2016, kernel_size

In [21]:
prompt = "The theory of relativity " * 256  # ~1280 tokens
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Capture both modes
results = {}
model.model.config.base_capacity = 512

for name, use_entropy in [("AdaKV", False), ("Entropy", True)]:
    model.model.config.use_entropy = use_entropy
    for layer in model.model.layers:
        if hasattr(layer.self_attn, 'kv_cluster'):
            del layer.self_attn.kv_cluster
    
    with torch.no_grad():
        model(**inputs, use_cache=True)
    
    h = model.model.layers[0].self_attn.kv_cluster
    results[name] = h.head_lens.tolist()
    print(f"{name}: {h.head_lens}  (total={h.klen_sum})")


Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pool

RecursionError: maximum recursion depth exceeded

In [16]:
import inspect
import adaptive_snapkv.monkeypatch.snapkv_utils as su

# Check if the new methods exist in the loaded module
print("has compute_renyi_entropy:", hasattr(su.AdaptiveSnapKVCluster, 'compute_renyi_entropy'))
print("has compute_entropy_budget:", hasattr(su.AdaptiveSnapKVCluster, 'compute_entropy_budget'))

# If False, the module needs to be reloaded
if not hasattr(su.AdaptiveSnapKVCluster, 'compute_entropy_budget'):
    import importlib
    importlib.reload(su)

has compute_renyi_entropy: True
has compute_entropy_budget: True


In [27]:
import torch
import torch.nn.functional as F
import adaptive_snapkv.monkeypatch.snapkv_utils as su
import math

# Re-apply the fixed calcul_attn_sore (no importlib.reload!)
def patched_calcul_attn_sore(self, key_states, query_states):
    bsz, num_heads, q_len, head_dim = query_states.shape
    
    if getattr(self, 'full_kv_scoring', False):
        q_slice = query_states
    else:
        q_slice = query_states[..., -self.window_size:, :]
    
    attn_weights = torch.matmul(q_slice, key_states.transpose(2, 3)) / math.sqrt(head_dim)
    
    n_q = attn_weights.shape[-2]
    n_k = min(attn_weights.shape[-1], self.window_size)
    
    # Fixed causal mask — broadcasts to [n_q, n_k]
    mask = torch.full((n_q, n_k), torch.finfo(attn_weights.dtype).min,
                      device=attn_weights.device)
    row_idx = torch.arange(n_q, device=attn_weights.device).unsqueeze(1)
    col_idx = torch.arange(n_k, device=attn_weights.device).unsqueeze(0)
    mask.masked_fill_(col_idx <= (row_idx + n_k - n_q), 0)
    
    attention_mask = mask[None, None, :, :]
    attn_weights[:, :, -n_q:, -n_k:] += attention_mask
    
    attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
    
    if getattr(self, 'use_entropy', False):
        p_alpha = attn_weights ** self.entropy_alpha
        head_entropy = torch.log(p_alpha.sum(dim=-1) + 1e-10) / (1 - self.entropy_alpha)
        head_entropy = head_entropy.mean(dim=-1)
    
    attn_weights_mean = attn_weights[:, :, -self.window_size:, : -self.window_size]
    if attn_weights_mean.shape[-1] == 0:
        empty = torch.zeros(bsz, num_heads, 0, device=attn_weights.device, dtype=attn_weights.dtype)
        return (empty, head_entropy) if getattr(self, 'use_entropy', False) else empty
    attn_weights_mean = attn_weights_mean.mean(dim=-2)
    
    if self.gqa_support:
        g = self.num_key_value_groups
        attn_weights_mean = attn_weights_mean.view(bsz, num_heads // g, g, -1)
        if self.gqa_func == 'max':
            attn_weights_mean = attn_weights_mean.max(dim=-2).values
        else:
            attn_weights_mean = attn_weights_mean.mean(dim=-2)
        if getattr(self, 'use_entropy', False):
            he = head_entropy.view(bsz, -1, g)
            head_entropy = he.max(dim=-1).values if self.gqa_func == 'max' else he.mean(dim=-1)
    
    if attn_weights_mean.shape[-1] >= self.kernel_size:
        p = self.kernel_size // 2
        if self.pooling == 'avgpool':
            attn_weights_mean = F.avg_pool1d(attn_weights_mean, self.kernel_size, padding=p, stride=1)
        else:
            attn_weights_mean = F.max_pool1d(attn_weights_mean, self.kernel_size, padding=p, stride=1)
    
    if getattr(self, 'use_entropy', False):
        return attn_weights_mean, head_entropy
    return attn_weights_mean

# Apply to class (affects new instances)
su.AdaptiveSnapKVCluster.calcul_attn_sore = patched_calcul_attn_sore

In [28]:

prompt = "The theory of relativity " * 256
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

model.model.config.use_entropy = True
for layer in model.model.layers:
    if hasattr(layer.self_attn, 'kv_cluster'):
        del layer.self_attn.kv_cluster

with torch.no_grad():
    model(**inputs, use_cache=True)

h = model.model.layers[0].self_attn.kv_cluster
print(f"Head lengths: {h.head_lens}  (total={h.klen_sum})")

Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True


L0: entropy=[4.40625, 5.0, 5.53125, 4.75, 4.15625, 5.34375, 5.03125, 0.85546875]
L0: budgets=[240, 308, 1232, 308, 240, 1232, 308, 240]
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
L1: entropy=[1.75, 2.015625, 3.859375, 1.625, 1.65625, 2.0625, 2.890625, 2.4375]
L1: budgets=[240, 308, 1232, 240, 240, 308, 1232, 308]
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
L2: entropy=[0.55859375, 1.125, 1.5625, 1.1875, 1.3515625, 1.8515625, 1.3984375, 1.9375]
L2: budgets=[240, 240, 308, 240, 308, 1232, 308, 1232]
Compress config(Ada): window_size=32, base_capacity=480, kernel_size=7, pooling=maxpool, floor_alpha=0.5, pyram_mode=False, beta=20, entropy=True
L3: entropy=[0.6171875, 1.6015625, 0.85546875, 0.9921875, 1.9609375, 1.6328125, 1.0390625, 2.953125]
L3: budgets=[240, 308, 240, 240, 1232, 308, 3